In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from pyspark.sql import functions as F

# Check dataset
print("Total records:", f"{cic_df.count():,}")
print("Total columns:", len(cic_df.columns))

# Check required columns
assert "Label" in cic_df.columns, "Label column missing"
assert "_source_file" in cic_df.columns, "Source file column missing"

print("Visualization environment ready!")

In [0]:
label_counts = (
    cic_df
    .groupBy("Label")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
)

label_counts["Label"] = label_counts["Label"].fillna("MISSING")

fig, ax = plt.subplots(figsize=(13, 8))

plot_data = label_counts.sort_values("count")

bars = ax.barh(
    plot_data["Label"],
    plot_data["count"]
)

ax.set_xscale("log")
ax.set_title("CIC-IDS2017: Traffic Class Distribution",
             fontsize=16, pad=15)
ax.set_xlabel("Number of Network Flows (Log Scale)")
ax.set_ylabel("Traffic Class")
ax.grid(axis="x", alpha=0.3)

for bar, value in zip(bars, plot_data["count"]):
    ax.text(
        value * 1.05,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,}",
        va="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [0]:
binary_counts = (
    cic_df
    .withColumn(
        "binary_label",
        F.when(
            F.upper(F.trim(F.col("Label"))) == "BENIGN",
            "BENIGN"
        ).otherwise("ATTACK")
    )
    .groupBy("binary_label")
    .count()
    .toPandas()
)

fig, ax = plt.subplots(figsize=(8, 7))

ax.pie(
    binary_counts["count"],
    labels=binary_counts["binary_label"],
    autopct="%1.2f%%",
    startangle=90,
    wedgeprops={"width": 0.42, "edgecolor": "white"},
    textprops={"fontsize": 12}
)

ax.set_title(
    "CIC-IDS2017: Benign vs Malicious Traffic",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql import functions as F
from functools import reduce

BASE_PATH = "/Volumes/workspace/default/cic_ids2017/raw"

files = sorted([
    f.path
    for f in dbutils.fs.ls(BASE_PATH)
    if f.name.lower().endswith(".csv")
])

dataframes = []

for path in files:

    filename = path.split("/")[-1]

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "PERMISSIVE")
        .csv(path)
    )

    # Normalize column names
    cleaned_columns = [
        c.strip().replace("\ufeff", "")
        for c in df.columns
    ]

    df = df.toDF(*cleaned_columns)

    # Preserve source filename
    df = df.withColumn("_source_file", F.lit(filename))

    # Normalize attack labels
    if "Label" in df.columns:
        df = df.withColumn(
            "Label",
            F.trim(F.col("Label"))
        )

    dataframes.append(df)

# Combine all files, preserving differences in columns
cic_df = reduce(
    lambda x, y: x.unionByName(
        y, allowMissingColumns=True
    ),
    dataframes
)

cic_df.createOrReplaceTempView("cic_ids2017")

print("Dataset successfully loaded!")
print("Columns:", len(cic_df.columns))
print("Rows:", f"{cic_df.count():,}")

In [0]:
label_distribution = (
    cic_df
    .groupBy("Label")
    .count()
    .orderBy(F.desc("count"))
)

display(label_distribution)

In [0]:
from pyspark.sql import functions as F

# Multiclass distribution with percentages
total = cic_df.count()

class_distribution = (
    cic_df
    .groupBy("Label")
    .agg(F.count("*").alias("count"))
    .withColumn(
        "percentage",
        F.round(F.col("count") * 100 / total, 4)
    )
    .orderBy(F.desc("count"))
)

display(class_distribution)

In [0]:
binary_df = cic_df.withColumn(
    "binary_label",
    F.when(
        F.upper(F.trim(F.col("Label"))) == "BENIGN",
        "BENIGN"
    ).otherwise("ATTACK")
)

display(
    binary_df.groupBy("binary_label")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
source_counts = (
    cic_df
    .groupBy("_source_file")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
)

source_counts["file"] = source_counts["_source_file"].str.replace(
    ".pcap_ISCX.csv", "", regex=False
)

fig, ax = plt.subplots(figsize=(13, 7))

plot_data = source_counts.sort_values("count")

bars = ax.barh(
    plot_data["file"],
    plot_data["count"]
)

ax.set_title("Network Traffic Records by Dataset File",
             fontsize=15, pad=15)
ax.set_xlabel("Number of Network Flows")
ax.set_ylabel("Capture Period")
ax.grid(axis="x", alpha=0.3)

for bar, value in zip(bars, plot_data["count"]):
    ax.text(
        value * 1.01,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,}",
        va="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [0]:
# Get actual attack counts from CIC-IDS2017
attack_counts = (
    cic_df
    .filter(
        F.col("Label").isNotNull() &
        (F.upper(F.trim(F.col("Label"))) != "BENIGN")
    )
    .groupBy("Label")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .toPandas()
)

# Calculate percentages
attack_percentages = attack_counts.copy()

total_attacks = attack_percentages["count"].sum()

attack_percentages["percentage"] = (
    attack_percentages["count"] / total_attacks * 100
)

attack_percentages = attack_percentages.sort_values(
    "percentage",
    ascending=True
)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))

bars = ax.barh(
    attack_percentages["Label"],
    attack_percentages["percentage"]
)

ax.set_title(
    "Contribution of Each Attack Type to Malicious Traffic",
    fontsize=15,
    pad=15
)

ax.set_xlabel("Percentage of All Malicious Flows (%)")
ax.set_ylabel("Attack Type")
ax.grid(axis="x", alpha=0.3)

for bar, value in zip(
    bars, attack_percentages["percentage"]
):
    ax.text(
        value + 0.1,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}%",
        va="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

print("Number of attack categories:", len(attack_counts))
print("Total malicious flows:", f"{total_attacks:,}")

In [0]:
from pyspark.sql import functions as F
import pandas as pd

# Columns to inspect
columns = cic_df.columns

expressions = []

for i, column in enumerate(columns):

    c = F.col(f"`{column.replace('`', '``')}`")
    value = F.lower(F.trim(c.cast("string")))

    # Null and empty values
    expressions.append(
        F.sum(
            F.when(
                c.isNull() | (value == ""),
                1
            ).otherwise(0)
        ).alias(f"missing_{i}")
    )

    # NaN and infinite values
    expressions.append(
        F.sum(
            F.when(
                value.isin(
                    "nan", "infinity", "-infinity",
                    "+infinity", "inf", "-inf", "+inf"
                ),
                1
            ).otherwise(0)
        ).alias(f"nonfinite_{i}")
    )

# One Spark aggregation
quality_result = cic_df.agg(*expressions).first().asDict()

# Construct report
quality_report = []

for i, column in enumerate(columns):

    missing = quality_result[f"missing_{i}"] or 0
    nonfinite = quality_result[f"nonfinite_{i}"] or 0

    quality_report.append({
        "Feature": column,
        "Missing Values": missing,
        "Non-Finite Values": nonfinite,
        "Missing (%)": round(missing / cic_df.count() * 100, 4)
    })

quality_df = pd.DataFrame(quality_report)

# Display only problematic columns
problematic = quality_df[
    (quality_df["Missing Values"] > 0) |
    (quality_df["Non-Finite Values"] > 0)
].sort_values(
    ["Missing Values", "Non-Finite Values"],
    ascending=False
)

print("Total features inspected:", len(columns))
print("Features with data quality issues:", len(problematic))

display(problematic)

In [0]:
if not problematic.empty:

    plot_data = problematic.copy()

    plot_data["Total Issues"] = (
        plot_data["Missing Values"] +
        plot_data["Non-Finite Values"]
    )

    plot_data = plot_data.sort_values(
        "Total Issues",
        ascending=True
    )

    fig, ax = plt.subplots(figsize=(12, 7))

    ax.barh(
        plot_data["Feature"],
        plot_data["Total Issues"]
    )

    ax.set_title(
        "CIC-IDS2017: Data Quality Issues by Feature",
        fontsize=15
    )

    ax.set_xlabel("Number of Invalid Records")
    ax.set_ylabel("Network Feature")
    ax.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    plt.show()

else:
    print("No missing or non-finite values detected.")

In [0]:
from pyspark.sql import functions as F

features = ["Flow Bytes/s", "Flow Packets/s"]

nonfinite_tokens = [
    "nan", "infinity", "+infinity", "-infinity",
    "inf", "+inf", "-inf"
]

conditions = {}

for feature in features:
    value = F.lower(F.trim(F.col(f"`{feature}`")))

    conditions[feature] = value.isin(*nonfinite_tokens)

    print(f"\n===== {feature} =====")

    display(
        cic_df
        .filter(conditions[feature])
        .groupBy(F.lower(F.trim(F.col(f"`{feature}`"))).alias("Value"))
        .count()
        .orderBy(F.desc("count"))
    )

# Check overlap between affected records
bytes_invalid = conditions["Flow Bytes/s"]
packets_invalid = conditions["Flow Packets/s"]

print("\n===== OVERLAP ANALYSIS =====")

print("Invalid Flow Bytes/s:",
      cic_df.filter(bytes_invalid).count())

print("Invalid Flow Packets/s:",
      cic_df.filter(packets_invalid).count())

print("Invalid in BOTH:",
      cic_df.filter(bytes_invalid & packets_invalid).count())

print("Invalid in EITHER:",
      cic_df.filter(bytes_invalid | packets_invalid).count())

In [0]:
from pyspark.sql import functions as F

# Exclude the filename metadata column
comparison_columns = [
    c for c in cic_df.columns
    if c != "_source_file"
]

# Count original records
total_records = cic_df.count()

# Count unique records across all original columns
unique_records = (
    cic_df
    .select(*comparison_columns)
    .dropDuplicates()
    .count()
)

# Calculate duplicates
duplicate_records = total_records - unique_records

duplicate_percentage = (
    duplicate_records / total_records * 100
)

print("========== DUPLICATE ANALYSIS ==========")
print("Total records:", f"{total_records:,}")
print("Unique records:", f"{unique_records:,}")
print("Duplicate records:", f"{duplicate_records:,}")
print("Duplicate percentage:", f"{duplicate_percentage:.4f}%")

In [0]:
from pyspark.sql import functions as F

# Compare all original fields except source filename
comparison_columns = [
    c for c in cic_df.columns
    if c != "_source_file"
]

# Identify groups of exactly identical records
duplicate_groups = (
    cic_df
    .groupBy(*comparison_columns)
    .agg(
        F.count("*").alias("occurrences"),
        F.collect_set("_source_file").alias("source_files")
    )
    .filter(F.col("occurrences") > 1)
)

# Overall duplicate analysis
duplicate_summary = duplicate_groups.agg(
    F.count("*").alias("Duplicate Groups"),
    F.sum("occurrences").alias("Records in Duplicate Groups"),
    F.sum(F.col("occurrences") - 1).alias("Redundant Records"),
    F.sum(
        F.when(F.size("source_files") > 1, 1).otherwise(0)
    ).alias("Groups Across Multiple Files")
)

display(duplicate_summary)

In [0]:
cross_file_duplicates = (
    duplicate_groups
    .filter(F.size("source_files") > 1)
    .groupBy("Label")
    .agg(
        F.count("*").alias("Cross_File_Duplicate_Groups"),
        F.sum(F.col("occurrences") - 1)
         .alias("Redundant_Records")
    )
    .orderBy(F.desc("Cross_File_Duplicate_Groups"))
)

display(cross_file_duplicates)

In [0]:
from pyspark.sql import functions as F

# Original feature and label columns
comparison_columns = [
    c for c in cic_df.columns
    if c != "_source_file"
]

# Identify identical records
duplicate_groups = (
    cic_df
    .groupBy(*comparison_columns)
    .agg(
        F.count("*").alias("occurrences"),
        F.collect_set("_source_file").alias("source_files")
    )
    .filter(F.col("occurrences") > 1)
)

# Calculate duplicate statistics for each label
duplicate_by_label = (
    duplicate_groups
    .groupBy("Label")
    .agg(
        F.count("*").alias("Duplicate_Groups"),
        F.sum("occurrences").alias("Records_in_Duplicate_Groups"),
        F.sum(F.col("occurrences") - 1).alias("Redundant_Records"),
        F.sum(
            F.when(
                F.size("source_files") > 1, 1
            ).otherwise(0)
        ).alias("Cross_File_Groups")
    )
    .orderBy(F.desc("Redundant_Records"))
)

display(duplicate_by_label)

In [0]:
from pyspark.sql import functions as F

# Exclude Label and source filename
feature_columns = [
    c for c in cic_df.columns
    if c not in ["Label", "_source_file"]
]

# Group by identical feature values
label_conflicts = (
    cic_df
    .groupBy(*feature_columns)
    .agg(
        F.countDistinct("Label").alias("number_of_labels"),
        F.collect_set("Label").alias("labels"),
        F.count("*").alias("total_occurrences")
    )
    .filter(F.col("number_of_labels") > 1)
)

print(
    "Number of feature combinations with conflicting labels:",
    label_conflicts.count()
)

display(label_conflicts.limit(20))

In [0]:
from pyspark.sql import functions as F

conflict_summary = label_conflicts.agg(
    F.count("*").alias("Conflicting Feature Groups"),

    F.sum("total_occurrences").alias(
        "Total Records in Conflict Groups"
    ),

    F.sum(F.col("total_occurrences") - 1).alias(
        "Excess Occurrences"
    ),

    F.max("number_of_labels").alias(
        "Maximum Labels per Feature Group"
    )
)

display(conflict_summary)

In [0]:
conflict_combinations = (
    label_conflicts
    .withColumn(
        "Label Combination",
        F.concat_ws(" + ", F.sort_array("labels"))
    )
    .groupBy("Label Combination")
    .agg(
        F.count("*").alias("Feature Groups"),
        F.sum("total_occurrences").alias("Total Records")
    )
    .orderBy(F.desc("Feature Groups"))
)

display(conflict_combinations)

In [0]:
binary_conflicts = (
    label_conflicts
    .withColumn(
        "Conflict Type",
        F.when(
            F.array_contains("labels", "BENIGN"),
            "BENIGN vs ATTACK"
        ).otherwise("ATTACK vs ATTACK")
    )
    .groupBy("Conflict Type")
    .agg(
        F.count("*").alias("Feature Groups"),
        F.sum("total_occurrences").alias("Total Records")
    )
    .orderBy(F.desc("Feature Groups"))
)

display(binary_conflicts)

In [0]:
import matplotlib.pyplot as plt
import numpy as np

important_features = [
    "Flow Duration",
    "Flow IAT Mean",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Packet Length Mean",
    "Packet Length Std",
    "Total Fwd Packets",
    "Total Backward Packets",
    "SYN Flag Count",
    "FIN Flag Count"
]

for feature in important_features:

    feature_index = available_features.index(feature)
    feature_col = f"feature_{feature_index}"

    benign = sample_pdf.loc[
        sample_pdf["binary_label"] == "BENIGN",
        feature_col
    ].dropna()

    attack = sample_pdf.loc[
        sample_pdf["binary_label"] == "ATTACK",
        feature_col
    ].dropna()

    benign = benign[np.isfinite(benign)]
    attack = attack[np.isfinite(attack)]

    # Signed logarithmic transformation for visualization
    benign = np.sign(benign) * np.log1p(np.abs(benign))
    attack = np.sign(attack) * np.log1p(np.abs(attack))

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.boxplot(
        [benign, attack],
        tick_labels=["BENIGN", "ATTACK"],
        showfliers=False
    )

    ax.set_title(
        f"Distribution Comparison: {feature}",
        fontsize=14
    )

    ax.set_ylabel("Signed Log1p Value")
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()